In [ ]:
import streamlit as st
import pandas as pd
from textblob import TextBlob
import plotly.express as px
import datetime
import io

# Optional: Speech recognition for converting voice to text
try:
    import speech_recognition as sr
    HAS_SR = True
except ImportError:
    HAS_SR = False

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="Student Voice Feedback System", page_icon="🎤", layout="wide")

# --- INITIALIZE SESSION STATE ---
# We use session state to store data temporarily while the app is running.
# In a real production app, you would save this to a database (e.g., SQLite, PostgreSQL, Firebase).
if 'feedbacks' not in st.session_state:
    st.session_state['feedbacks'] = pd.DataFrame(columns=[
        'Timestamp', 'Student_Name', 'Roll_Number', 'Event_Name',
        'Photo', 'Transcript', 'Sentiment', 'Polarity'
    ])

# Helper function to load sample data for demonstration purposes
def load_sample_data():
    sample_data = pd.DataFrame([
        {'Timestamp': datetime.datetime.now(), 'Student_Name': 'Alice Smith', 'Roll_Number': '101', 'Event_Name': 'Hackathon 2026', 'Photo': None, 'Transcript': 'The event was absolutely amazing and I learned so much about AI.', 'Sentiment': 'Positive', 'Polarity': 0.8},
        {'Timestamp': datetime.datetime.now(), 'Student_Name': 'Bob Jones', 'Roll_Number': '102', 'Event_Name': 'Hackathon 2026', 'Photo': None, 'Transcript': 'It was poorly organized and the internet kept dropping. Very frustrating.', 'Sentiment': 'Negative', 'Polarity': -0.6},
        {'Timestamp': datetime.datetime.now(), 'Student_Name': 'Charlie Brown', 'Roll_Number': '103', 'Event_Name': 'Science Fair', 'Photo': None, 'Transcript': 'It was okay, nothing special but decent overall.', 'Sentiment': 'Neutral', 'Polarity': 0.05},
        {'Timestamp': datetime.datetime.now(), 'Student_Name': 'Alice Smith', 'Roll_Number': '101', 'Event_Name': 'Science Fair', 'Photo': None, 'Transcript': 'Great presentations, really enjoyed the chemistry experiments!', 'Sentiment': 'Positive', 'Polarity': 0.6},
    ])
    st.session_state['feedbacks'] = pd.concat([st.session_state['feedbacks'], sample_data], ignore_index=True)

# --- UTILITY FUNCTIONS ---
def analyze_sentiment(text):
    """Analyze text sentiment using TextBlob"""
    if not text:
        return "Neutral", 0.0

    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return "Positive", polarity
    elif polarity < -0.1:
        return "Negative", polarity
    else:
        return "Neutral", polarity

def transcribe_audio(audio_bytes):
    """Transcribe audio using SpeechRecognition"""
    if not HAS_SR:
        return "SpeechRecognition library not installed. Showing placeholder text."

    recognizer = sr.Recognizer()
    try:
        # Convert streamlit audio bytes to a format SpeechRecognition can read
        with sr.AudioFile(io.BytesIO(audio_bytes)) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data)
            return text
    except sr.UnknownValueError:
        return "Audio could not be understood. Please try speaking clearer."
    except Exception as e:
        return f"Could not transcribe audio. Error: {str(e)}"

# --- MAIN APP ROUTING ---
st.title("🎤 Student Event Voice Feedback System")

# Sidebar Navigation
page = st.sidebar.radio("Navigate", ["Record Feedback", "Analysis Dashboard"])

if st.sidebar.button("Load Sample Data"):
    load_sample_data()
    st.sidebar.success("Sample data loaded!")

# ==========================================
# PAGE 1: RECORD FEEDBACK
# ==========================================
if page == "Record Feedback":
    st.header("Submit Your Feedback")

    with st.form("feedback_form", clear_on_submit=True):
        col1, col2 = st.columns(2)

        with col1:
            st.subheader("1. User Details")
            student_name = st.text_input("Full Name *")
            roll_number = st.text_input("Roll Number / ID *")

        with col2:
            st.subheader("2. Event Details")
            event_name = st.selectbox("Select Event *",
                                      ["Hackathon 2026", "Science Fair", "Annual Sports Meet", "Tech Symposium", "Other"])
            if event_name == "Other":
                event_name = st.text_input("Specify Event Name")

        st.subheader("3. Photo Identity")
        # Allow either camera input or file upload
        photo_method = st.radio("Choose photo method", ["Upload Image", "Take Photo"], horizontal=True)
        photo_data = None
        if photo_method == "Upload Image":
            photo_data = st.file_uploader("Upload your photo", type=['jpg', 'jpeg', 'png'])
        else:
            photo_data = st.camera_input("Take a picture")

        st.subheader("4. Voice Feedback")
        st.info("**Please answer the following questions in your recording:**\n"
                "1. What did you enjoy most about this event?\n"
                "2. Were there any challenges or things you disliked?\n"
                "3. Would you recommend this event to others?")

        # Audio Recording Widget (Available in Streamlit >= 1.37)
        audio_value = st.audio_input("Record your feedback here")

        # Fallback text area in case audio isn't working for the user
        fallback_text = st.text_area("Or type your feedback manually (if audio is unavailable)", height=100)

        submitted = st.form_submit_button("Submit Feedback", type="primary")

        if submitted:
            if not student_name or not roll_number or not event_name:
                st.error("Please fill in all mandatory fields (*).")
            elif not audio_value and not fallback_text:
                st.error("Please provide either voice feedback or text feedback.")
            else:
                with st.spinner("Processing feedback..."):
                    transcript = ""
                    if audio_value:
                        # Convert audio to text
                        transcript = transcribe_audio(audio_value.getvalue())
                    else:
                        transcript = fallback_text

                    # Calculate Sentiment
                    sentiment_label, polarity_score = analyze_sentiment(transcript)

                    # Save to dataframe
                    new_record = pd.DataFrame([{
                        'Timestamp': datetime.datetime.now(),
                        'Student_Name': student_name,
                        'Roll_Number': roll_number,
                        'Event_Name': event_name,
                        'Photo': photo_data, # Keeping the uploaded file object
                        'Transcript': transcript,
                        'Sentiment': sentiment_label,
                        'Polarity': polarity_score
                    }])

                    st.session_state['feedbacks'] = pd.concat([st.session_state['feedbacks'], new_record], ignore_index=True)

                st.success(f"Feedback submitted successfully! Detected Sentiment: **{sentiment_label}**")
                with st.expander("View Transcript"):
                    st.write(transcript)

# ==========================================
# PAGE 2: ANALYSIS DASHBOARD
# ==========================================
elif page == "Analysis Dashboard":
    st.header("📊 Feedback Analysis Dashboard")

    df = st.session_state['feedbacks']

    if df.empty:
        st.warning("No feedback data available yet. Please record some feedback or load sample data from the sidebar.")
    else:
        # --- OVERALL SUMMARY ---
        st.subheader("Overall Summary")
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Total Feedbacks", len(df))
        m2.metric("Total Unique Students", df['Roll_Number'].nunique())
        m3.metric("Total Events", df['Event_Name'].nunique())

        avg_polarity = df['Polarity'].mean()
        overall_sentiment = "Positive" if avg_polarity > 0.1 else ("Negative" if avg_polarity < -0.1 else "Neutral")
        m4.metric("Avg Sentiment", overall_sentiment, f"{avg_polarity:.2f} Score")

        # --- TABS FOR DETAILED ANALYSIS ---
        tab1, tab2, tab3 = st.tabs(["Event-Wise Analysis", "Student-Wise Analysis", "Raw Data"])

        # TAB 1: EVENT-WISE
        with tab1:
            col_e1, col_e2 = st.columns([1, 2])

            with col_e1:
                selected_event = st.selectbox("Select Event to Analyze", df['Event_Name'].unique())
                event_df = df[df['Event_Name'] == selected_event]

                st.write(f"**Total Feedbacks for {selected_event}:** {len(event_df)}")
                st.write(f"**Average Polarity:** {event_df['Polarity'].mean():.2f}")

            with col_e2:
                # Sentiment Distribution Pie Chart
                sentiment_counts = event_df['Sentiment'].value_counts().reset_index()
                sentiment_counts.columns = ['Sentiment', 'Count']
                fig_pie = px.pie(sentiment_counts, values='Count', names='Sentiment',
                                 title=f"Sentiment Distribution - {selected_event}",
                                 color='Sentiment',
                                 color_discrete_map={'Positive': 'green', 'Negative': 'red', 'Neutral': 'gray'})
                st.plotly_chart(fig_pie, use_container_width=True)

            # Show specific feedback items for this event
            st.write("#### Recent Feedback Highlights")
            for idx, row in event_df.head(5).iterrows():
                emoji = "🟢" if row['Sentiment'] == "Positive" else "🔴" if row['Sentiment'] == "Negative" else "⚪"
                st.info(f"{emoji} **{row['Student_Name']} ({row['Roll_Number']})**: \"{row['Transcript']}\"")

        # TAB 2: STUDENT-WISE
        with tab2:
            student_list = df.apply(lambda x: f"{x['Student_Name']} ({x['Roll_Number']})", axis=1).unique()
            selected_student_str = st.selectbox("Search / Select Student", student_list)

            # Extract roll number from the string
            selected_roll = selected_student_str.split("(")[-1].replace(")", "")
            student_df = df[df['Roll_Number'] == selected_roll]

            st.markdown("---")
            col_s1, col_s2 = st.columns([1, 3])

            with col_s1:
                # Display Photo if available
                # Get the most recent photo
                recent_photo = student_df.iloc[-1]['Photo']
                if recent_photo is not None:
                    st.image(recent_photo, caption="Student Photo", use_container_width=True)
                else:
                    st.image("https://api.dicebear.com/7.x/initials/svg?seed="+student_df.iloc[-1]['Student_Name'], caption="No Photo Uploaded")

            with col_s2:
                st.subheader(student_df.iloc[-1]['Student_Name'])
                st.write(f"**Roll Number:** {selected_roll}")
                st.write(f"**Total Events Attended / Reviewed:** {len(student_df)}")

                # Student's Feedback History
                st.write("#### Feedback History")
                for idx, row in student_df.iterrows():
                    color = "green" if row['Sentiment'] == 'Positive' else "red" if row['Sentiment'] == 'Negative' else "gray"
                    st.markdown(f"**Event:** {row['Event_Name']} | **Date:** {row['Timestamp'].strftime('%Y-%m-%d')}")
                    st.markdown(f"**Sentiment:** :{color}[{row['Sentiment']}]")
                    st.caption(f"\"{row['Transcript']}\"")
                    st.markdown("---")

        # TAB 3: RAW DATA
        with tab3:
            st.subheader("Complete Data Log")
            # Drop the photo column for display purposes as it contains binary objects
            display_df = df.drop(columns=['Photo']).copy()
            # Format timestamp
            display_df['Timestamp'] = display_df['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
            st.dataframe(display_df, use_container_width=True)

            # Allow CSV download
            csv = display_df.to_csv(index=False).encode('utf-8')
            st.download_button(
                label="Download Data as CSV",
                data=csv,
                file_name='feedback_analysis_export.csv',
                mime='text/csv',
            )